In [4]:
import sys
import os
from dotenv import load_dotenv, find_dotenv
import geopandas as gpd
import matplotlib.pyplot as plt
import fiona
from shapely.geometry import box
import pandas as pd

load_dotenv(find_dotenv())

DATA_PATH = os.getenv("DATA_PATH")
DEWEY_PATH = os.getenv("DEWEY_PATH")

In [5]:
# Create a list of all .parquet files in DEWEY_PATH
parquet_files = [f for f in os.listdir(DEWEY_PATH) if f.endswith('.parquet')]

In [ ]:
summ_dfs = []
for f in parquet_files:
    df = pd.read_parquet(os.path.join(DEWEY_PATH, f))
    df['FILE_DATE'] = pd.to_datetime(df['FILE_DATE'], errors='coerce')
    df['PERMIT_DATE'] = pd.to_datetime(df['PERMIT_DATE'], errors='coerce')
    df['FINAL_DATE'] = pd.to_datetime(df['FINAL_DATE'], errors='coerce')
    summ_df = df.groupby(['JURISDICTION', 'STATE']).agg(
        count = ('JURISDICTION', 'count'),  # any rows 
        first_file_date = ('FILE_DATE', 'min'), 
        last_file_date = ('FILE_DATE', 'max'),
        missing_file_date = ('FILE_DATE', lambda x: x.isna().sum()),
        first_permit_date = ('PERMIT_DATE', 'min'),
        last_permit_date = ('PERMIT_DATE', 'max'),
        missing_permit_date = ('PERMIT_DATE', lambda x: x.isna().sum()),
        first_final_date = ('FINAL_DATE', 'min'),
        last_final_date = ('FINAL_DATE', 'max'),
        missing_final_date = ('FINAL_DATE', lambda x: x.isna().sum()),
        missing_status = ('STATUS_NORMALIZED', lambda x: x.isna().sum())
    ).reset_index()
    summ_df['filename'] = f
    summ_dfs.append(summ_df)

summ_df = pd.concat(summ_dfs)
summ_df = summ_df.sort_values(by=['STATE', 'JURISDICTION'], ascending=True).reset_index(drop=True)



In [7]:
summ_df.to_csv(os.path.join(DATA_PATH, "dewey_summary_with_filenames.csv"), index=False)
summ_df.to_parquet(os.path.join(DATA_PATH, "dewey_summary_with_filenames.parquet"), index=False)


In [8]:
summ_df2 = summ_df.groupby(['JURISDICTION', 'STATE']).agg(
    count = ('count', 'sum'),
    first_file_date = ('first_file_date', 'min'),
    last_file_date = ('last_file_date', 'max'),
    missing_file_date = ('missing_file_date', 'sum'),
    first_permit_date = ('first_permit_date', 'min'),
    last_permit_date = ('last_permit_date', 'max'),
    missing_permit_date = ('missing_permit_date', 'sum'),
    first_final_date = ('first_final_date', 'min'),
    last_final_date = ('last_final_date', 'max'),
    missing_final_date = ('missing_final_date', 'sum'),
    missing_status = ('missing_status', 'sum')
).reset_index()

summ_df2 = summ_df2.sort_values(by=['STATE', 'JURISDICTION'], ascending=True).reset_index(drop=True)

summ_df2.to_csv(os.path.join(DATA_PATH, "dewey_summary.csv"), index=False)
summ_df2.to_parquet(os.path.join(DATA_PATH, "dewey_summary.parquet"), index=False)


